In [ ]:
import os
import numpy as np
import pandas as pd
import gc
import time
from contextlib import contextmanager
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, PolynomialFeatures, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import zipfile 
from pathlib import Path

import missingno as msno


In [2]:
zip_path = Path("C:\\Users\\chris\\Initiez_vous_au_ML_Ops\\Projet+Mise+en+prod+-+home-credit-default-risk.zip")
extract_dir = Path("data")

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

print("Extraction terminée !")

Extraction terminée !


In [3]:
pos_cash_balance = pd.read_csv(r"C:\Users\chris\Initiez_vous_au_ML_Ops\data\POS_CASH_balance.csv")
print('pos cash data shape: ', pos_cash_balance.shape)
pos_cash_balance.head()

pos cash data shape:  (10001358, 8)


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [ ]:
%matplotlib inline
msno.matrix(pos_cash_balance.sample(10000))

In [6]:
pos_cash_balance.duplicated().sum()

np.int64(0)

In [ ]:
pos_cash_balance.isnull().sum()

SK_ID_PREV                   0
SK_ID_CURR                   0
MONTHS_BALANCE               0
CNT_INSTALMENT           26071
CNT_INSTALMENT_FUTURE    26087
NAME_CONTRACT_STATUS         0
SK_DPD                       0
SK_DPD_DEF                   0
dtype: int64

In [9]:
pos_cash_balance.describe()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,SK_DPD,SK_DPD_DEF
count,1.000136e+07,1.000136e+07,1.000136e+07,9.975287e+06,9.975271e+06,1.000136e+07,1.000136e+07
mean,1.903217e+06,2.784039e+05,-3.501259e+01,1.708965e+01,1.048384e+01,1.160693e+01,6.544684e-01
std,5.358465e+05,1.027637e+05,2.606657e+01,1.199506e+01,1.110906e+01,1.327140e+02,3.276249e+01
min,1.000001e+06,1.000010e+05,-9.600000e+01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.434405e+06,1.895500e+05,-5.400000e+01,1.000000e+01,3.000000e+00,0.000000e+00,0.000000e+00
50%,1.896565e+06,2.786540e+05,-2.800000e+01,1.200000e+01,7.000000e+00,0.000000e+00,0.000000e+00
75%,2.368963e+06,3.674290e+05,-1.300000e+01,2.400000e+01,1.400000e+01,0.000000e+00,0.000000e+00
max,2.843499e+06,4.562550e+05,-1.000000e+00,9.200000e+01,8.500000e+01,4.231000e+03,3.595000e+03


In [21]:
pos_cash_balance['NAME_CONTRACT_STATUS'].value_counts();

In [15]:
# Historique
pos_cash_balance['MONTHS_BALANCE'].value_counts();

In [17]:
# Nombre d’échéances prévues
pos_cash_balance['CNT_INSTALMENT'].value_counts();

In [19]:
# Nombre d’échéances restantes
pos_cash_balance['CNT_INSTALMENT_FUTURE'].value_counts();

In [ ]:
# SK_DPD: Days past due. The number of days that the payment of the installment was overdue: 
# The value 0 means that the payment was made on time, while a positive value indicates the number of days the payment was late.
pos_cash_balance['SK_DPD'].value_counts();

In [12]:
# SK_DPD_DEF: Days past due with penalty:
# tracks the number of days that a payment was overdue and has incurred a penalty or fine as a result. 
# A value of 0 indicates that the payment was made on time without any penalties, 
# while a positive value indicates the number of days the payment was late and has resulted in a penalty.
pos_cash_balance['SK_DPD_DEF'].value_counts();

In [ ]:
pos_cash_balance.drop(columns=['NAME_CONTRACT_STATUS'], inplace=True)

In [23]:
# pas d'encodage en l'absence de données non numériques
pos_cash_balance_agg = pos_cash_balance.groupby("SK_ID_PREV").agg(["mean", "min", "max", "sum", "size"])
pos_cash_balance_agg.columns = ["POS_" + "_".join(col).upper() for col in pos_cash_balance_agg.columns]
pos_cash_balance_agg.reset_index(inplace=True)


In [24]:
pos_agg = pos_cash_balance_agg.merge(
    pos_cash_balance[["SK_ID_PREV", "SK_ID_CURR"]].drop_duplicates(),
    on="SK_ID_PREV",
    how="left"
)


In [26]:
pos_final = pos_agg.groupby("SK_ID_CURR").agg(["mean", "min", "max", "sum"])
pos_final.columns = ["POS_FINAL_" + "_".join(col).upper() for col in pos_final.columns]
pos_final.reset_index(inplace=True)
pos_final.head()


,SK_ID_CURR,POS_FINAL_SK_ID_PREV_MEAN,POS_FINAL_SK_ID_PREV_MIN,POS_FINAL_SK_ID_PREV_MAX,POS_FINAL_SK_ID_PREV_SUM,POS_FINAL_POS_SK_ID_CURR_MEAN_MEAN,POS_FINAL_POS_SK_ID_CURR_MEAN_MIN,POS_FINAL_POS_SK_ID_CURR_MEAN_MAX,POS_FINAL_POS_SK_ID_CURR_MEAN_SUM,POS_FINAL_POS_SK_ID_CURR_MIN_MEAN,...,POS_FINAL_POS_SK_DPD_DEF_MAX_MAX,POS_FINAL_POS_SK_DPD_DEF_MAX_SUM,POS_FINAL_POS_SK_DPD_DEF_SUM_MEAN,POS_FINAL_POS_SK_DPD_DEF_SUM_MIN,POS_FINAL_POS_SK_DPD_DEF_SUM_MAX,POS_FINAL_POS_SK_DPD_DEF_SUM_SUM,POS_FINAL_POS_SK_DPD_DEF_SIZE_MEAN,POS_FINAL_POS_SK_DPD_DEF_SIZE_MIN,POS_FINAL_POS_SK_DPD_DEF_SIZE_MAX,POS_FINAL_POS_SK_DPD_DEF_SIZE_SUM
0,100001,1.610838e+06,1369693,1851984,3221677,100001.0,100001.0,100001.0,200002.0,100001.0,...,7,7,3.5,0,7,7,4.500000,4,5,9
1,100002,1.038818e+06,1038818,1038818,1038818,100002.0,100002.0,100002.0,100002.0,100002.0,...,0,0,0.0,0,0,0,19.000000,19,19,19
2,100003,2.281150e+06,1810518,2636178,6843451,100003.0,100003.0,100003.0,300009.0,100003.0,...,0,0,0.0,0,0,0,9.333333,8,12,28
3,100004,1.564014e+06,1564014,1564014,1564014,100004.0,100004.0,100004.0,100004.0,100004.0,...,0,0,0.0,0,0,0,4.000000,4,4,4
4,100005,2.495675e+06,2495675,2495675,2495675,100005.0,100005.0,100005.0,100005.0,100005.0,...,0,0,0.0,0,0,0,11.000000,11,11,11
